# Robust identifier system showcase

This notebook documents the identifier philosophy that the rest of the semantic math stack relies on. The core idea is simple:

> **Canonical names are plain validated strings. Display LaTeX is derived from those names, not used as the identity itself.**

That separation is what makes round-tripping, testing, and MathLive transport predictable.

## Setup

We only expose the repository source tree to the notebook kernel. All examples below use the canonical import path `gu_toolkit.identifiers`, then later import `gu_toolkit.Symbolic` to show the same rules at a higher level.

In [ ]:
import import_setup

In [ ]:
import sympy as sp

from gu_toolkit.identifiers import (
    IdentifierError,
    function_head_to_latex,
    identifier_to_latex,
    parse_identifier,
    register_symbol_latex,
    render_latex,
    symbol,
    validate_identifier,
)

## 1. Canonical spelling rules

A canonical identifier must start with a letter and then use only letters, digits, and underscores.

Two underscore conventions matter:

- a single underscore separates atoms and usually means a subscript boundary
- a double underscore preserves a literal underscore inside one atom

That is why `theta_x` and `theta__x` are both valid but mean different things.

In [ ]:
good_names = ['x', 'theta', 'theta_x', 'theta__x', 'a_1_2']
good_summary = []
for name in good_names:
    assert validate_identifier(name) == name
    good_summary.append((name, identifier_to_latex(name)))

good_summary

## 2. Parsing and rendering should be reversible

If a user sees an explicit display form such as `\mathrm{velocity}` or `a_{1,2}`, the backend should be able to recover the original canonical spelling.

This is why the low-level API includes both renderers and parsers.

In [ ]:
round_trip_examples = {
    'velocity': r'\mathrm{velocity}',
    'theta__x': r'\mathrm{theta\_x}',
    'a_1_2': r'a_{1,2}',
    'Force_t': r'\operatorname{Force}_{t}',
}

for canonical, latex in round_trip_examples.items():
    renderer = identifier_to_latex if canonical[0].islower() else function_head_to_latex
    assert renderer(canonical) == latex
    assert parse_identifier(latex) == canonical

expr = symbol('velocity') + symbol('theta__x')
render_latex(expr)

## 3. Bad identifiers fail fast

The identifier layer should reject malformed names immediately so later code never has to guess whether a string was a LaTeX fragment, a typo, or a real canonical identifier.

In [ ]:
bad_names = [r'\theta', '1x', 'x y', 'a-b']
bad_messages = {}
for name in bad_names:
    try:
        validate_identifier(name)
    except IdentifierError as exc:
        bad_messages[name] = str(exc)
    else:
        raise AssertionError(f'Expected {name!r} to fail validation.')

bad_messages

## 4. Display overrides stay separate from identity

Sometimes you want a custom display form such as `\vec{v}` for a symbol that still has the canonical backend name `velocity`.

The override API exists for presentation only; it does not change the stored identifier.

In [ ]:
velocity = symbol('velocity')
register_symbol_latex(velocity, r'\vec{v}')

latex_with_override = render_latex(velocity + symbol('x'))
assert r'\vec{v}' in latex_with_override
assert velocity.name == 'velocity'

latex_with_override

## 5. Higher-level symbolic helpers enforce the same rules

The low-level validator is not the only guardrail. The convenience APIs in `gu_toolkit.Symbolic` should refuse the same bad names so notebook users cannot accidentally bypass the contract.

This is important because those helpers are often the first place users create symbols and indexed families.

In [ ]:
from gu_toolkit.Symbolic import FunctionFamily, SymbolFamily, symbols

a = SymbolFamily('a')
force_x = FunctionFamily('Force__x')
assert a[1, 2].name == 'a_1_2'
assert force_x.name == 'Force__x'
assert symbols('theta__x').name == 'theta__x'

symbolic_failures = {}
for label, callback in {
    'SymbolFamily(\\theta)': lambda: SymbolFamily(r'\theta'),
    'FunctionFamily(1Force)': lambda: FunctionFamily('1Force'),
    'symbols(\\theta)': lambda: symbols(r'\theta'),
    'symbols(1x)': lambda: symbols('1x'),
}.items():
    try:
        callback()
    except IdentifierError as exc:
        symbolic_failures[label] = str(exc)
    else:
        raise AssertionError(f'Expected {label} to raise IdentifierError.')

symbolic_failures

## Refactoring rule of thumb

When future work touches this area, keep asking two questions:

1. **What is the canonical identifier that the backend stores?**
2. **How is the display form derived from that identifier at the boundary?**

If a proposed refactor mixes those two concerns back together, it is moving away from the design this notebook documents.